In [2]:
from docling.document_converter import DocumentConverter
import pandas as pd

In [3]:
import torch
import torchvision

print(torch.__version__)
print(torchvision.__version__)

2.4.0+cpu
0.19.0+cpu


In [4]:
# Конфиг
LMSTUDIO_BASE_URL = "http://localhost:1234/v1"
TARGET_MODEL = "qwen2.5-vl-7b-instruct"   # укажи точное имя VLM из LM Studio (важно: с 'vl')

# Утилиты печати
def ok(msg):    print(f"✅ {msg}")
def warn(msg):  print(f"⚠️ {msg}")
def err(msg):   print(f"❌ {msg}")

In [5]:
import json, urllib.request, urllib.error

def http_get(url, timeout=5):
    req = urllib.request.Request(url, method="GET")
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.getcode(), r.read()

try:
    code, body = http_get(f"{LMSTUDIO_BASE_URL}/models")
    if 200 <= code < 300:
        ok(f"LM Studio сервер отвечает: {code}")
        data = json.loads(body.decode("utf-8"))
        models = [m["id"] for m in data.get("data", [])]
        print("Доступные модели:", models or "нет")
    else:
        err(f"Сервер ответил кодом {code}")
        models = []
except urllib.error.URLError as e:
    err(f"Не удалось подключиться к {LMSTUDIO_BASE_URL}: {e}")
    models = []

✅ LM Studio сервер отвечает: 200
Доступные модели: ['qwen2.5-14b-instruct-1m', 'text-embedding-nomic-embed-text-v1.5']


In [6]:
# === Импорты и базовые настройки ===
import os, io, base64, traceback, logging
from pathlib import Path
from typing import List, Tuple
import pandas as pd
from PIL import Image

# Прогресс-бары (работают в Jupyter)
from tqdm.auto import tqdm

# Пути (твои)
in_folder  = r"C:\Users\w-w3\YandexDisk\А3\convert_markdown\docs_in"
out_folder = r"C:\Users\w-w3\YandexDisk\А3\convert_markdown\docs_out"
Path(out_folder).mkdir(parents=True, exist_ok=True)

# Глушим «болтливый» лог библиотек
for noisy in ["docling", "rapidocr", "onnxruntime", "PIL"]:
    logging.getLogger(noisy).setLevel(logging.ERROR)
logging.getLogger().setLevel(logging.WARNING)

# Расширения
DOC_EXT = {".pdf", ".docx", ".pptx", ".html", ".htm", ".txt", ".png", ".jpg", ".jpeg", ".tif", ".tiff"}
TAB_EXT = {".csv", ".tsv", ".xls", ".xlsx"}
ALL_EXT = DOC_EXT | TAB_EXT

## Клиент LM Studio + функции для описаний (таблица → Markdown)

In [8]:
# === LM Studio (OpenAI-совместимый сервер) ===
from openai import OpenAI
LMSTUDIO_BASE_URL = "http://localhost:1234/v1"         # LM Studio → Developer → Start Server
VLM_MODEL = "qwen2.5-vl-7b-instruct"                   # именно VL (vision) модель
client = OpenAI(base_url=LMSTUDIO_BASE_URL, api_key="lm-studio")

def is_markdown_table(text: str) -> bool:
    """Грубый детектор Markdown-таблицы."""
    if not text:
        return False
    lines = [ln.rstrip() for ln in text.splitlines() if ln.strip()]
    if len(lines) < 2: 
        return False
    has_pipe = any(line.startswith("|") and line.endswith("|") for line in lines[:2])
    has_sep  = any("---" in line and "|" in line for line in lines[:3])
    return has_pipe and has_sep

def describe_image_or_table(img_path: Path) -> str:
    """
    Если на изображении таблица — возвращаем чистую Markdown-таблицу.
    Иначе — краткое русское описание графика/картинки.
    """
    try:
        b64 = base64.b64encode(img_path.read_bytes()).decode("utf-8")
        prompt = (
            "Определи, содержит ли изображение таблицу с ячейками.\n"
            "Если ЭТО ТАБЛИЦА: перепиши аккуратно как Markdown-таблицу и верни ТОЛЬКО таблицу без текста до/после.\n"
            "Если НЕ таблица: верни краткое описание по-русски (тип графика/диаграммы, оси, тренды) в 1–3 предложениях."
        )
        resp = client.chat.completions.create(
            model=VLM_MODEL,
            messages=[{
                "role":"user",
                "content":[
                    {"type":"text","text": prompt},
                    {"type":"image_url","image_url":{"url": f"data:image/png;base64,{b64}"}}
                ]
            }],
            temperature=0.1,
            max_tokens=800,
        )
        return (resp.choices[0].message.content or "").strip()
    except Exception as e:
        return f"(не удалось получить результат от модели: {e})"

## Docling: конвертер с picture-description для PDF

In [9]:
# === Docling конвертер с picture-description для PDF ===
from docling.document_converter import DocumentConverter
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, PictureDescriptionApiOptions
from docling.document_converter import PdfFormatOption

try:
    from docling_core.types.doc.document import PictureDescriptionData
except Exception:
    PictureDescriptionData = None


def build_pdf_converter_with_enrichments() -> DocumentConverter:
    """
    Конвертер без создания images/tables папок.
    Только текст + описание изображений.
    """

    pp = PdfPipelineOptions(
        enable_remote_services=True,

        # ❗ отключаем ВСЕ артефакты
        generate_picture_images=False,
        generate_table_images=False,   # ← НОВОЕ (важно!)
    )

    # ❗ иногда docling всё равно создаёт директории — отключаем экспорт логически
    pp.do_picture_description = True

    pp.picture_description_options = PictureDescriptionApiOptions(
        url="http://localhost:1234/v1/chat/completions",
        params=dict(
            model=VLM_MODEL,
            max_completion_tokens=200,
            seed=42
        ),
        prompt="Опиши изображение кратко по-русски: тип графика/диаграммы, оси, тренды, ключевые элементы.",
        timeout=90,
    )

    return DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=pp)
        }
    )

## Вспомогательные функции (сохранения, замены <!-- image -->)

In [10]:
def ensure_dirs(root: Path) -> Tuple[Path, Path]:
    images_dir = root / "images"; images_dir.mkdir(parents=True, exist_ok=True)
    tables_dir = root / "tables"; tables_dir.mkdir(parents=True, exist_ok=True)
    return images_dir, tables_dir

def export_tables_from_doc(doc, tables_dir: Path) -> List[str]:
    """Сохраняем таблицы в CSV/HTML и отдаём MD-превью."""
    blocks = []
    tables = getattr(doc, "tables", [])
    for i, tbl in enumerate(tables, start=1):
        try:
            df: pd.DataFrame = tbl.export_to_dataframe()
            (tables_dir / f"table_{i}.csv").write_text(df.to_csv(index=False), encoding="utf-8")
            df.to_html(tables_dir / f"table_{i}.html", index=False)
            blocks.append(
                f"\n\n### Таблица {i}\n\n{df.to_markdown(index=False)}\n\n"
                f"> Файлы: [tables/table_{i}.csv](tables/table_{i}.csv), "
                f"[tables/table_{i}.html](tables/table_{i}.html)"
            )
        except Exception:
            try:
                html = tbl.export_to_html(doc=doc)
                (tables_dir / f"table_{i}.html").write_text(html, encoding="utf-8")
                blocks.append(f"\n\n### Таблица {i}\n\n(сохранена как HTML)\n")
            except Exception:
                blocks.append(f"\n\n> Таблица {i}: не удалось экспортировать.")
    return blocks

def extract_images_from_doc(doc, images_dir: Path) -> List[Path]:
    """Сохраняем изображения из doc.pictures/doc.figures."""
    saved: List[Path] = []
    pics = getattr(doc, "pictures", [])
    for i, pic in enumerate(pics, start=1):
        img = None
        if hasattr(pic, "image") and pic.image is not None:
            if hasattr(pic.image, "bytes") and pic.image.bytes:
                img = Image.open(io.BytesIO(pic.image.bytes))
            elif hasattr(pic.image, "pil_image") and pic.image.pil_image is not None:
                img = pic.image.pil_image
        if img is None:
            continue
        p = images_dir / f"image_{i}.png"
        img.convert("RGB").save(p, "PNG")
        saved.append(p)

    if not saved:  # fallback
        figs = getattr(doc, "figures", [])
        for j, fig in enumerate(figs, start=1):
            img = None
            if hasattr(fig, "image_bytes") and fig.image_bytes:
                img = Image.open(io.BytesIO(fig.image_bytes))
            elif hasattr(fig, "pil_image") and fig.pil_image is not None:
                img = fig.pil_image
            if img is None:
                continue
            p = images_dir / f"image_f_{j}.png"
            img.convert("RGB").save(p, "PNG")
            saved.append(p)
    return saved

def captions_from_enrichments(doc) -> List[str]:
    """Подписи из Docling-enrichment (если он сработал)."""
    caps = []
    pics = getattr(doc, "pictures", [])
    for pic in pics:
        cap = ""
        for ann in getattr(pic, "annotations", []) or []:
            if PictureDescriptionData and isinstance(ann, PictureDescriptionData):
                cap = ann.text or ""
                break
            if hasattr(ann, "text"):
                cap = ann.text or ""
                break
        caps.append(cap)
    return caps

def replace_image_markers_with_blocks(md: str, img_files: List[str], captions: List[str]) -> str:
    """
    На месте <!-- image -->:
      - если caption — Markdown-таблица, вставляем таблицу;
      - иначе — картинку + подпись.
    """
    if "<!-- image -->" not in md or not img_files:
        return md

    parts = md.split("<!-- image -->")
    rebuilt = parts[0]
    for idx, tail in enumerate(parts[1:], start=1):
        i = min(idx-1, len(img_files)-1)
        fname = img_files[i]
        cap = captions[i] if i < len(captions) else ""
        if is_markdown_table(cap):
            block = f"\n\n{cap}\n\n> Исходник: `images/{fname}`\n\n"
        else:
            block = f'\n\n![{fname}](images/{fname})\n\n*Подпись:* {cap}\n\n'
        rebuilt += block + tail
    return rebuilt

## Конвертеры: табличные файлы и «общий» через Docling (с прогрессом по шагам)

In [11]:
def convert_tabular_file(src: Path, out_dir: Path) -> str:
    images_dir, tables_dir = ensure_dirs(out_dir)
    ext = src.suffix.lower()
    if ext in {".csv", ".tsv"}:
        sep = "," if ext == ".csv" else "\t"
        df = pd.read_csv(src, sep=sep)
    else:
        df = pd.read_excel(src, sheet_name=0)

    (tables_dir / f"{src.stem}.csv").write_text(df.to_csv(index=False), encoding="utf-8")
    df.to_html(tables_dir / f"{src.stem}.html", index=False)

    md = (
        f"# {src.name}\n\n## Таблица\n\n{df.to_markdown(index=False)}\n\n"
        f"> Файлы: [tables/{src.stem}.csv](tables/{src.stem}.csv), "
        f"[tables/{src.stem}.html](tables/{src.stem}.html)\n"
    )
    return md

def convert_doc_file(src: Path, out_dir: Path, pdf_converter: DocumentConverter) -> str:
    """
    Этапы с прогрессом:
      1) конвертация Docling
      2) экспорт таблиц
      3) извлечение картинок
      4) подписи (enrichment/LM Studio)
      5) сборка Markdown
    """
    images_dir, tables_dir = ensure_dirs(out_dir)
    steps = ["Docling convert", "Export tables", "Extract images", "Captions", "Assemble MD"]
    p = tqdm(total=len(steps), desc=f"{src.name}", leave=False)

    # 1) Конвертация
    try:
        conv = pdf_converter if src.suffix.lower() == ".pdf" else DocumentConverter()
        res = conv.convert(str(src))
        doc = res.document
        p.update(1)
    except Exception as e:
        p.close()
        raise RuntimeError(f"Docling convert failed: {e}")

    # 2) Таблицы
    try:
        table_blocks = export_tables_from_doc(doc, tables_dir)
        p.update(1)
    except Exception as e:
        p.close()
        raise RuntimeError(f"Export tables failed: {e}")

    # 3) Картинки
    try:
        img_paths = extract_images_from_doc(doc, images_dir)
        img_files = [x.name for x in img_paths]
        p.update(1)
    except Exception as e:
        p.close()
        raise RuntimeError(f"Extract images failed: {e}")

    # 4) Подписи
    try:
        captions: List[str] = []
        if src.suffix.lower() == ".pdf":
            captions = captions_from_enrichments(doc)
            if len(captions) < len(img_files):
                captions += [""] * (len(img_files) - len(captions))
            for i, fname in enumerate(img_files):
                if not captions[i]:
                    captions[i] = describe_image_or_table(images_dir / fname)
        else:
            for fname in img_files:
                captions.append(describe_image_or_table(images_dir / fname))
        p.update(1)
    except Exception as e:
        p.close()
        raise RuntimeError(f"Captions failed: {e}")

    # 5) Сборка Markdown (подмена <!-- image -->)
    try:
        md = doc.export_to_markdown()
        md = replace_image_markers_with_blocks(md, img_files, captions)
        if img_files and "<!-- image -->" not in md:
            md += "\n\n## Иллюстрации\n"
            for fname, cap in zip(img_files, captions):
                if is_markdown_table(cap):
                    md += f"\n\n{cap}\n\n> Исходник: `images/{fname}`\n"
                else:
                    md += f"\n![{fname}](images/{fname})\n\n*Подпись:* {cap}\n"
        if table_blocks:
            md += "\n\n## Таблицы\n" + "\n".join(table_blocks)
        p.update(1); p.close()
        return md
    except Exception as e:
        p.close()
        raise RuntimeError(f"Assemble MD failed: {e}")

## Оркестрация по папке + главный запуск (большой прогресс)

In [ ]:
def convert_one(src: Path, out_root: Path, pdf_converter: DocumentConverter) -> Path:
    out_dir = out_root / src.stem
    out_dir.mkdir(parents=True, exist_ok=True)
    try:
        if src.suffix.lower() in TAB_EXT:
            md = convert_tabular_file(src, out_dir)
        elif src.suffix.lower() in DOC_EXT:
            md = convert_doc_file(src, out_dir, pdf_converter)
        else:
            md = f"# {src.name}\n\n> Пропущен: неподдерживаемый формат."
    except Exception as e:
        md = f"# {src.name}\n\n> Ошибка: {e}\n\n```\n{traceback.format_exc()}\n```"

    # 💡 здесь исправлено имя выходного файла:
    out_md = out_dir / f"{src.stem}.md"
    out_md.write_text(md, encoding="utf-8")
    return out_md

def convert_folder(in_dir: Path, out_dir: Path, recursive: bool = False):
    assert in_dir.is_dir(), f"{in_dir} не папка"
    files = [p for p in (in_dir.rglob("*") if recursive else in_dir.iterdir())
             if p.is_file() and p.suffix.lower() in ALL_EXT]
    if not files:
        print("⚠️ В папке нет поддерживаемых файлов."); return

    print(f"🗂 Найдено файлов: {len(files)}")
    pdf_conv = build_pdf_converter_with_enrichments()

    pbar = tqdm(total=len(files), desc="Всего файлов", ncols=100)
    for f in files:
        _ = convert_one(f, out_dir, pdf_conv)
        pbar.set_postfix_str(f.name)
        pbar.update(1)
    pbar.close()
    print("✅ Готово.")

# === Запуск ===
convert_folder(Path(in_folder), Path(out_folder), recursive=False)

🗂 Найдено файлов: 4


Всего файлов:   0%|                                                           | 0/4 [00:00<?, ?it/s]

HouseHold. Ручная отправка уведомлений о новом лс_деактивации лс_Confluence.pdf:   0%|          | 0/5 [00:00<?…

[INFO] 2026-04-22 17:49:07,137 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-04-22 17:49:07,145 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-04-22 17:49:07,147 [RapidOCR] download_file.py:68: Initiating download: https://www.modelscope.cn/models/RapidAI/RapidOCR/resolve/v3.8.0/torch/PP-OCRv4/det/ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-04-22 17:49:08,523 [RapidOCR] download_file.py:82: Download size: 13.83MB
[INFO] 2026-04-22 17:49:11,202 [RapidOCR] download_file.py:95: Successfully saved to: C:\python_envs\docling_env\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-04-22 17:49:11,207 [RapidOCR] main.py:50: Using C:\python_envs\docling_env\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.pth
[INFO] 2026-04-22 17:49:11,516 [RapidOCR] base.py:22: Using engine_name: torch
[INFO] 2026-04-22 17:49:11,516 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-04-22 17:49:11,518 [RapidOCR] download_file.py:68: Initiating downl

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

[WARNING] 2026-04-22 17:50:14,434 [RapidOCR] main.py:132: The text detection result is empty
[WARNING] 2026-04-22 17:50:15,584 [RapidOCR] main.py:132: The text detection result is empty
[WARNING] 2026-04-22 17:50:18,790 [RapidOCR] main.py:132: The text detection result is empty
[WARNING] 2026-04-22 17:50:25,178 [RapidOCR] main.py:132: The text detection result is empty


OAPI_HouseHold_v1.1.0_МЕТОДИЧЕСКОЕ ПОСОБИЕ ПО ИНТЕГРАЦИИ ДЛЯ ПАРТНЕРОВ.pdf:   0%|          | 0/5 [00:00<?, ?it…

[WARNING] 2026-04-22 17:50:31,044 [RapidOCR] main.py:132: The text detection result is empty
[WARNING] 2026-04-22 17:50:35,119 [RapidOCR] main.py:132: The text detection result is empty
[WARNING] 2026-04-22 17:50:45,064 [RapidOCR] main.py:132: The text detection result is empty
[WARNING] 2026-04-22 17:50:46,936 [RapidOCR] main.py:132: The text detection result is empty
[WARNING] 2026-04-22 17:50:49,564 [RapidOCR] main.py:132: The text detection result is empty
[WARNING] 2026-04-22 17:50:50,800 [RapidOCR] main.py:132: The text detection result is empty
[WARNING] 2026-04-22 17:50:54,631 [RapidOCR] main.py:132: The text detection result is empty
[WARNING] 2026-04-22 17:50:55,850 [RapidOCR] main.py:132: The text detection result is empty
[WARNING] 2026-04-22 17:50:58,391 [RapidOCR] main.py:132: The text detection result is empty
[WARNING] 2026-04-22 17:51:01,055 [RapidOCR] main.py:132: The text detection result is empty
[WARNING] 2026-04-22 17:51:05,029 [RapidOCR] main.py:132: The text det

OAPI_Подписки_Уведомления_на_поставщиков_v1_1_5 МЕТОДИЧЕСКОЕ ПОСОБИЕ ПО ИНТЕГРАЦИИ ДЛЯ ПАРТНЕРОВ.pdf:   0%|   …

[WARNING] 2026-04-22 17:54:31,175 [RapidOCR] main.py:132: The text detection result is empty
[WARNING] 2026-04-22 17:54:38,154 [RapidOCR] main.py:132: The text detection result is empty
[WARNING] 2026-04-22 17:54:49,491 [RapidOCR] main.py:132: The text detection result is empty
Stage preprocess failed for run 3, pages [63]: std::bad_alloc
Stage preprocess failed for run 3, pages [64]: std::bad_alloc
Stage preprocess failed for run 3, pages [65]: std::bad_alloc
Stage preprocess failed for run 3, pages [66]: std::bad_alloc
Stage preprocess failed for run 3, pages [67]: std::bad_alloc
Stage preprocess failed for run 3, pages [68]: std::bad_alloc
Stage preprocess failed for run 3, pages [69]: std::bad_alloc
Stage preprocess failed for run 3, pages [70]: std::bad_alloc
Stage preprocess failed for run 3, pages [71]: std::bad_alloc
Stage preprocess failed for run 3, pages [72]: std::bad_alloc
Stage preprocess failed for run 3, pages [73]: std::bad_alloc
Stage preprocess failed for run 3, page